# Growth (GRO) — Risk-Factor Construction Spec (USE4-faithful)

**Purpose.** This notebook is the **source-of-truth spec** for building the
**GRO** (Growth) style factor exactly as MSCI describes it in *Barra USE4 Empirical
Notes* (Appendix A, p. 52). It is **not** a research notebook. The deliverable
is a clean, USE4-faithful GRO factor written to parquet, suitable for input to a
multi-factor risk model.

**Audience.** You — plus whatever AI assistant you are driving. Treat its output as deeply untrustworthy until audited. Follow the stages linearly. Each stage has:
- ✅ **PDF SPEC** — exact verbatim quote or paraphrase from the USE4 documentation
- ❓ **NOT IN PDF** — something the PDF does not specify; a judgment call you must make
- 💡 **DEFAULT** — a reasonable default for any ❓ item, chosen to match standard Barra practice
- 🧪 **VALIDATE** — sanity checks to run after each stage

**Inputs:**
- Sharadar SF1 annual fundamentals (`EPS`, `SPS` per ticker per fiscal year)
- Sharadar SEP/daily prices for ESTU membership and market cap
- ESTU monthly panel (built from Sharadar SEP + TICKERS)

**Deliverable:** A parquet file `gro_use4.parquet` keyed by
`(permaticker, signal_date)` containing the standardised GRO exposure for
every stock in the coverage universe on every monthly signal date.

**Companion specs:**
- `02_style_factors/eyld/eyld_spec.ipynb` — Earnings Yield (similar SF1 pipeline)
- `02_style_factors/beta/beta_spec.ipynb` — Beta (no dependency, but related Volatility family)


## §1. The USE4 GRO spec — verbatim quotes

### 1a. Empirical Notes, Appendix A (p. 52) — formal descriptor definition

> **Growth (GRO)**
>
> *Definition:* `GRO = 0.70 · EGRLF + 0.20 · EGRO + 0.10 · SGRO`
>
> | Symbol | Name | Definition |
> |--------|------|------------|
> | **EGRLF** | Long-term predicted earnings growth | Analyst-forecasted 3–5 year earnings growth rate (sell-side consensus) |
> | **EGRO** | Earnings growth (trailing 5 years) | Normalized OLS slope of annual EPS regressed against time over 5 fiscal years |
> | **SGRO** | Sales growth (trailing 5 years) | Normalized OLS slope of annual SPS regressed against time over 5 fiscal years |

### 1b. Methodology Notes, Section 2.3 (p. 9) — standardisation rule

> *"Descriptors are standardised to have a mean of 0 and a standard deviation of 1.
> ... μ_l is the cap-weighted mean of the descriptor (within the estimation universe),
> and σ_l is the equal-weighted standard deviation."*

### 1c. Methodology Notes, Section 2.2 (p. 8) — outlier treatment

> *"We trim these observations to three standard deviations from the mean."*

---

**That is all the PDFs say about GRO construction.**

The PDF does **not** specify:
- How to handle negative mean EPS/SPS in the normalisation denominator
- What to do when fewer than 5 years of data are available
- Which data source or filing-date filter to use for PIT compliance
- The treatment of EGRLF when analyst forecast data is unavailable
- Whether to renormalise weights when one descriptor is missing


## §2. End-to-end pipeline

```
┌─────────────────────────────────────────────────────────────────────┐
│  STAGE 0:  Setup, imports, paths, parameters                        │
├─────────────────────────────────────────────────────────────────────┤
│  STAGE 1:  Load SF1 annual fundamentals (EPS, SPS)                  │
├─────────────────────────────────────────────────────────────────────┤
│  STAGE 2:  Build ESTU per signal_date  (mcap, in_estu)              │
├─────────────────────────────────────────────────────────────────────┤
│  STAGE 4:  GRO estimator                                            │
│            Per stock × signal_date:                                 │
│              EGRO = β̂_EPS / mean(EPS),  5-yr OLS, PIT-filtered     │
│              SGRO = β̂_SPS / mean(SPS),  5-yr OLS, PIT-filtered     │
│              GRO  = 0.90·EGRO + 0.10·SGRO  (proxied weights)       │
├─────────────────────────────────────────────────────────────────────┤
│  STAGE 5:  Outlier treatment  (3σ trim, cap-weighted mean ± 3σ)     │
├─────────────────────────────────────────────────────────────────────┤
│  STAGE 6:  Standardise  (cap-weighted mean=0, EW std=1)             │
├─────────────────────────────────────────────────────────────────────┤
│  STAGE 7:  Save deliverable                                         │
├─────────────────────────────────────────────────────────────────────┤
│  STAGE 8:  Validate                                                 │
└─────────────────────────────────────────────────────────────────────┘
```

**PIT discipline:** For any `signal_date t`, the only data permitted is SF1 records
with `datekey ≤ t`. No daily price data is used — this is a pure fundamental factor.

**Note:** Stage 3 (cap-weighted market benchmark) is omitted for GRO because the
estimator does not require excess returns or a market return series.


## §3. Output schema — what the worker delivers

**Raw descriptor column:** `gro`
**Standardised score column:** `gro_score`

**File:** `data/out/gro_use4.parquet`

**Compression:** zstd, statistics=True

**Schema:**

| Column | Type | Description |
|---|---|---|
| `permaticker` | Int64 | Sharadar permanent ticker ID |
| `signal_date` | Date | End-of-month rebalance date (last trading day of month) |
| `in_estu` | Bool | Whether this stock was in the estimation universe on this date |
| `mcap` | Float64 | Market cap on signal_date (used for cap-weighting) |
| `gro` | Float64 | **Raw GRO descriptor** — proxied composite (0.90·EGRO + 0.10·SGRO) |
| `gro_score` | Float64 | **Final GRO exposure** — standardised cross-sectionally (the deliverable) |
| `n_obs` | UInt32 | Number of annual fiscal-year observations used in the EGRO/SGRO regressions |

**Note on `n_obs`:** Unlike time-series factors where `n_obs` counts trading days, here
`n_obs` is the number of annual data points fed into the OLS regressions (range: 3–5).
Stocks are excluded if fewer than `MIN_OBS = 3` annual observations are available.

**Coverage rules:**
- Compute `gro` and `gro_score` for **every stock with sufficient data**
  (`n_obs ≥ MIN_OBS`), not just ESTU members.
- Standardisation moments (mean, std) are computed using **ESTU members only**.
- Non-ESTU stocks get the *same* standardisation parameters applied so the final
  factor is comparable across the coverage universe.


## §4. STAGE 0 — Setup, imports, paths

Standard imports. Use polars, not pandas, throughout (project convention).

✅ **PDF SPEC — GRO composite weights (Empirical Notes, Appendix A, p. 52):**

> `GRO_W_EGRLF = 0.70` — long-term predicted earnings growth (analyst forecast, 3–5 yr)
> `GRO_W_EGRO  = 0.20` — trailing earnings growth (5-yr normalized OLS slope of EPS)
> `GRO_W_SGRO  = 0.10` — trailing sales growth (5-yr normalized OLS slope of SPS)
> `GRO_YEARS   = 5`    — trailing fiscal years for EGRO and SGRO estimation

❓ **NOT IN PDF — EGRLF data source, minimum observations, PIT filter, and negative-mean handling.**
See §7 (Stage 4) for the full NOT-IN-PDF breakdown and defaults.

```python
# ───── Parameters ────────────────────────────────────────────────────────────
FACTOR_TYPE = "fundamental"     # uses SF1 annual data, not daily prices

# USE4 composite weights (Empirical Notes, Appendix A, p. 52)
GRO_W_EGRLF = 0.70    # long-term predicted EPS growth (analyst forecast)
GRO_W_EGRO  = 0.20    # trailing earnings growth (5-yr OLS slope of EPS)
GRO_W_SGRO  = 0.10    # trailing sales growth (5-yr OLS slope of SPS)

# 💡 DEFAULT (NOT IN PDF) — EGRLF proxy
# Analyst 3–5 yr forecasts are not available in Sharadar SF1.
# Proxy EGRLF with EGRO; effective weights become 0.90 · EGRO + 0.10 · SGRO.
# Set GRO_W_EGRLF_PROXY = 0.0 when real EGRLF data is available.
GRO_W_EGRLF_PROXY = GRO_W_EGRLF  # = 0.70; added to GRO_W_EGRO → effective weight 0.90

# 💡 DEFAULT (NOT IN PDF) — lookback window
GRO_YEARS = 5    # target: 5 fiscal years of annual EPS/SPS data

# 💡 DEFAULT (NOT IN PDF) — minimum annual observations for a valid regression
MIN_OBS = 3    # require ≥ 3 annual data points; fewer than 3 is a degenerate regression

# 💡 DEFAULT (NOT IN PDF) — EGRO/SGRO pre-clip
# The OLS normalization (slope / mean(EPS)) can produce astronomically large values
# when mean(EPS) is very small but positive (passes the > 0 guard). A ±500% annual
# growth cap is economically defensible and prevents 3σ trim contamination.
EGRO_CLIP = 5.0  # clip EGRO and SGRO to [-5, +5] before composite

# 💡 DEFAULT (NOT IN PDF) — calendar start
CALENDAR_START = date(1999, 1, 1)
```


## §5. STAGE 1 — Load SF1 annual fundamentals

GRO is a **pure fundamental factor** — it requires no daily price data beyond what is
needed for ESTU membership (Stage 2). The primary input is Sharadar SF1.

✅ **PDF SPEC:** GRO descriptors are computed from annual EPS (earnings per share) and SPS
(sales per share) as reported in company financial filings.

❓ **NOT IN PDF — which SF1 dimension to use.**
Sharadar SF1 has multiple dimensions:
- `ARY` — as-reported annual (original filing, no restatements applied retroactively)
- `MRY` — most-recent annual (latest restatement as of each datekey)

❓ **NOT IN PDF — PIT filter column.**
The `datekey` column in SF1 is the SEC filing date (the date the report became publicly
available). The `calendardate` column is the fiscal year-end date.

💡 **DEFAULT:** Use `dimension = 'ARY'` filtered by `datekey ≤ signal_date`. This gives
the strictest PIT compliance — each stock's annual EPS/SPS is only available after the
company filed its 10-K. Using `calendardate` instead would introduce look-ahead bias
(fiscal year ends months before the 10-K is filed).

**SF1 columns needed:**
| Column | Description |
|---|---|
| `permaticker` | Permanent ticker ID |
| `datekey` | SEC filing date (PIT filter) |
| `calendardate` | Fiscal year-end date (identifies which fiscal year) |
| `eps` | Diluted earnings per share |
| `sps` | Sales per share |

🧪 **VALIDATE:**
- SF1 covers tickers from the ESTU universe with `datekey` spanning the full history
- `eps` and `sps` are non-null for the majority of rows in `ARY` dimension
- No future `datekey` values relative to today appear in filtered data


## §6. STAGE 2 — Build ESTU per signal_date

**Identical to the shared USE4 ESTU construction** — see
`01_estu/estu_build.ipynb` for the full algorithm.

✅ **PDF SPEC:** ESTU membership is determined at each rebalance date based on market cap
thresholds, country filters, and data availability rules.

**Outputs used by GRO:**
- `estu_monthly`: `(permaticker, signal_date, in_estu, mcap)` — one row per stock per
  monthly signal date. `mcap` is the market cap at the signal date (used for
  cap-weighting in standardisation).

🧪 **VALIDATE:**
- ~2,500–3,500 ESTU members per date post-2005
- `mcap` is positive and non-null for all ESTU members


## §7. STAGE 4 — GRO estimator

✅ **PDF SPEC (Empirical Notes, Appendix A, p. 52):**

**Growth (GRO)** is a composite:

> `GRO = 0.70 · EGRLF + 0.20 · EGRO + 0.10 · SGRO`

**EGRO — Trailing earnings growth (5-year OLS slope, normalized):**

> The normalized OLS slope of annual EPS regressed against time index t = 1, ..., N
> over the trailing N fiscal years (target N = 5):
>
> `EGRO = β̂_EPS / mean(EPS)`
>
> where β̂_EPS is the OLS slope from `EPS_t = α + β · t, t = 1,...,N`.
> The normalization by mean(EPS) converts the absolute slope into a fractional growth rate
> (approximately the compound annual growth rate for small growth values).

**SGRO — Trailing sales growth (5-year OLS slope, normalized):**

> Identical to EGRO but using SPS (sales per share):
>
> `SGRO = β̂_SPS / mean(SPS)`

**EGRLF — Long-term predicted earnings growth:**

> Analyst-forecasted 3–5 year earnings growth rate (sell-side consensus I/B/E/S or
> FactSet LTG). Not available in Sharadar SF1.

❓ **NOT IN PDF — EGRLF data unavailability:** Sharadar does not include analyst consensus
long-term EPS growth forecasts. USE4 weights EGRLF at 0.70 — the dominant descriptor.

💡 **DEFAULT (EGRLF proxy):** Proxy EGRLF with EGRO (trailing 5-year growth as a
stand-in for predicted growth). Effective proxied weights:

```
GRO = (0.70 + 0.20) · EGRO + 0.10 · SGRO  =  0.90 · EGRO + 0.10 · SGRO
```

Document as a **known structural deviation** from USE4. Preserve the code structure
(separate EGRLF weight constant) so real EGRLF can be swapped in later.

❓ **NOT IN PDF — negative mean normalization:** If `mean(EPS) ≤ 0` (chronically
loss-making stock), the normalized slope EGRO is undefined or sign-inverted — a positive
trend in losses divided by a negative mean produces a negative EGRO, misleadingly
suggesting deteriorating growth.

💡 **DEFAULT:** Mark EGRO as `null` when `mean(EPS) ≤ 0`. Mark SGRO as `null` when
`mean(SPS) ≤ 0`. SPS is almost always positive for operating companies, but the guard
is important for completeness.

❓ **NOT IN PDF — minimum observations:** USE4 specifies a 5-year target but does not
state what to do when fewer than 5 years of data are available.

💡 **DEFAULT:** Accept ≥ 3 annual observations (`MIN_OBS = 3`). OLS on 3 points is
minimally stable; 2 or fewer points are degenerate (slope = exact rise/run, no
regression benefit). Record the actual number of observations used as `n_obs`.

❓ **NOT IN PDF — PIT filter:** The spec does not specify when annual report data becomes
available.

💡 **DEFAULT:** Filter SF1 to `datekey ≤ signal_date`, `dimension = 'ARY'`. Take the
N most recent distinct fiscal years (identified by `calendardate`) per ticker per
signal date. This ensures data is available only after the company files its 10-K.

❓ **NOT IN PDF — one descriptor null, other available:** If EGRO is computable but
SGRO is null (or vice versa), the composite weight sum is less than 1.0.

💡 **DEFAULT:** Renormalize available weights to sum to 1.0:
- Both available: `GRO = 0.90 · EGRO + 0.10 · SGRO`
- Only EGRO: `GRO = 1.00 · EGRO`
- Only SGRO: `GRO = 1.00 · SGRO`
- Both null: `GRO = null`

---
*The section above (PDF SPEC quote through final 💡 DEFAULT) is the `STAGE4_DESCRIPTION`
that `/build-factor` will inject verbatim into the Stage 4 stub docstring. Content below
this line is supplementary guidance for the implementer and is not extracted.*
---

**Implementation notes:**

- **Time index t:** Use integer rank (1 = oldest, N = most recent), not calendar year.
  This ensures the slope has units of EPS-per-year regardless of gaps in filing history.
- **OLS closed form:** For a regression of y against t = 1..N with intercept:
  ```
  β̂ = (N·Σ(t·y) − Σt·Σy) / (N·Σ(t²) − (Σt)²)
  ```
  Precompute Σt, Σt², Σy, Σ(ty) per ticker per signal_date using Polars group_by+agg.
- **Most recent N fiscal years:** Sort by `calendardate` descending, take top N rows per
  ticker. Ensure `datekey ≤ signal_date` filter is applied *before* taking the top N.
- **n_obs reporting:** `n_obs` = min(N_EPS, N_SPS) where N is the number of valid annual
  observations used. Report the smaller count (binding constraint).

🧪 **VALIDATE:**

- Spot-check a high-growth stock (e.g. NVDA in 2020–2023): `gro_score` should be in Q4–Q5
- Spot-check a mature low-growth stock (e.g. KO, JNJ): `gro_score` should be in Q1–Q3
- Raw `gro` descriptor cross-sectional range: approximately (−2, +5) — growth rates can
  be large and right-skewed (high-growth stocks)
- `n_obs` distribution: majority at 5, with a tail at 3–4 for newer listings
- Stocks with negative mean EPS are null: tech/biotech start-ups should be excluded from EGRO
- SGRO null rate should be much lower than EGRO null rate (SPS almost always positive)


## §8. STAGE 5 — Outlier treatment

❓ **NOT IN PDF for GRO specifically.** Methodology Notes Section 2.2 (p. 8) describes
a general outlier-treatment algorithm that applies to all descriptors:

> *"We trim these observations to three standard deviations from the mean."*

💡 **DEFAULT:** Apply 3σ trimming per `signal_date`, computed within ESTU using
**cap-weighted mean ± 3 × equal-weighted std**. Applied to all stocks in the coverage
universe after estimation.

**Note:** GRO descriptors are right-skewed (high-growth outliers can be very large).
The 3σ trim will clip more aggressively on the right tail than the left. This is
expected and correct behaviour.

🧪 **VALIDATE:**
- ~0.5–2% of ESTU rows should be clipped per date (expect slightly higher clip rate on
  the right tail due to growth skew)
- Post-trim distribution should be less skewed than raw `gro`
- Check that the trim bounds are sensible: `lo` ~ −1.0 to −3.0, `hi` ~ +3.0 to +8.0


## §9. STAGE 6 — Standardise (cap-weighted mean=0, EW std=1)

✅ **PDF SPEC (Methodology Notes §2.3, p. 9):**

> *"μ_l is the cap-weighted mean of the descriptor (within the estimation universe),
> and σ_l is the equal-weighted standard deviation."*

**Per `signal_date t`, using only ESTU members:**

```
μ_t  = Σ_{i ∈ ESTU_t} (mcap_i · gro_i) / Σ mcap_i    # cap-weighted mean
σ_t  = std_{i ∈ ESTU_t}(gro_i)                         # equal-weighted std
gro_score_{i,t} = (gro_i − μ_t) / σ_t                  # applied to ALL i
```

Three critical points:
1. Moments estimated on **ESTU only**; applied to **every stock** in the coverage universe.
2. Mean is **cap-weighted**; std is **equal-weighted**.
3. Cap-weighting the mean ensures the cap-weighted market portfolio has zero GRO exposure.

🧪 **VALIDATE:**
- `Σ_{i ∈ ESTU} (mcap_i · gro_score_i) / Σ mcap_i ≈ 0` (cap-weighted mean = 0 by construction)
- `std_{i ∈ ESTU}(gro_score_i) ≈ 1` (equal-weighted std = 1 by construction)


## §10. STAGE 7 — Save deliverable

Write the final panel to `data/out/gro_use4.parquet`.
Compression: zstd. Statistics: True.

Include `gro` (raw composite descriptor) for downstream diagnostics and attribution.
The downstream consumer will use `gro_score` (the standardised exposure).

**Schema assertion:** Verify all column types match §3 exactly before writing.
**Read-back assertion:** Re-read the parquet and confirm shape and column order.


## §11. STAGE 8 — Validation

### Required checks (all must pass before signing off)

**1a. Standardisation moments on ESTU — cap-weighted mean:**
```
|cap-weighted mean of gro_score (ESTU)| < 1e-6   (machine zero by construction)
```

**1b. Standardisation moments on ESTU — EW std:**
```
|EW std of gro_score (ESTU) − 1.0| < 0.02
```

**2. Raw descriptor sanity:**
```
Median raw gro cross-sectionally: approximately 0.05–0.30
(expected: most stocks have modest positive growth rates)
Max |gro| < 10.0  (extreme growth outliers clipped by Stage 5)
```

**3. Coverage stability:**
```
≥ 4,000 stocks with non-null gro_score per date post-2005
```

**4. Factor stability (month-over-month rank correlation):**
```
MoM Spearman ρ > 0.95
(annual earnings/sales data changes slowly — expect very high stability)
```

**5. Economic direction (quintile forward return):**
```
Q5 (high GRO) should exhibit higher forward 12-month returns than Q1 (low GRO)
in bull-market periods — growth stocks outperform in risk-on regimes
Spot checks:
  High GRO stocks (Q5): NVDA, TSLA, AMZN during growth periods
  Low GRO stocks (Q1): mature utilities, legacy industrials
```

**6. Disk vs memory equivalence:**
```
max abs diff of gro values between in-memory panel and parquet read-back < 1e-10
```

---

**Shared validation conventions (all factors, 2026-06-10):**
- **Coverage (check 3)** is evaluated on **completed months only** — the final
  signal date is the in-progress month (freshest data can lag the Sharadar
  refresh) and is excluded from the floor.
- **Fraction of scores in [−3, +3]** is reported for the full universe *and* for
  ESTU; the ESTU figure is the operationally relevant one (non-ESTU micro-caps
  pull the all-universe tail).
- Common check machinery lives in `common/ (your shared utilities)`
  .


## §12. Master list of ❓ NOT-IN-PDF decisions

| # | Decision | Default chosen | Alternative | Where to revisit |
|---|---|---|---|---|
| 1 | EGRLF data unavailability | Proxy with EGRO; effective weights 0.90·EGRO + 0.10·SGRO | Skip EGRLF contribution entirely (use 0.20·EGRO + 0.10·SGRO) | When analyst LTG forecast data (I/B/E/S, FactSet) becomes available in Sharadar |
| 2 | Negative mean EPS/SPS | Mark EGRO/SGRO null when mean(EPS) ≤ 0 or mean(SPS) ≤ 0 | Compute on absolute values, flip sign | When researching loss-making growth stocks (e.g. pre-profitability tech) |
| 3 | Minimum observations | Accept ≥ 3 annual data points (MIN_OBS = 3) | Require exactly 5 years (USE4 target) | If coverage is sufficient without the 3-year floor |
| 4 | PIT filter | `datekey ≤ signal_date`; annual series derived from ARQ (ARY absent) | `calendardate + 90d lag`, `MRY` | If ARY causes excessive data sparsity in early periods |
| 5 | One descriptor null | Renormalize available weights to sum to 1.0 | Mark GRO null if any component is null | If renormalized GRO is found to distort cross-sectional ranking |


## §13. Common pitfalls — read this before coding

**Pitfall 1: Using calendardate instead of datekey as the PIT filter**
Annual reports are filed months after fiscal year-end (typically 60–90 days for 10-K).
Using `calendardate ≤ signal_date` would include data that was not yet publicly available,
introducing look-ahead bias. Always filter on `datekey ≤ signal_date`.

**Pitfall 2: Negative mean sign flip in EGRO/SGRO**
If a stock has negative mean EPS over the 5-year window (chronically loss-making), the
normalized slope `β̂ / mean(EPS)` will have its sign flipped — a positive earnings trend
becomes a negative EGRO. This makes loss-making growth companies look like value traps.
Guard: null-out EGRO when `mean(EPS) ≤ 0`.

**Pitfall 3: Integer time index vs calendar year**
The time index t = 1,...,N must be a simple integer rank (1 = oldest fiscal year, N = most
recent). Using calendar year values (e.g. 2018, 2019, 2020) produces the same slope but
changes its scale, making the raw OLS β̂ much smaller. The normalization by mean(EPS)
corrects for absolute scale, but using rank avoids confusion about units.

**Pitfall 4: n_obs interpretation**
Unlike time-series factors (Beta, MOM, RESVOL) where `n_obs` counts trading days in a
rolling window, for GRO `n_obs` is the number of annual fiscal-year observations used
in the OLS regression. The max value is 5, not 252 or 504. Downstream consumers of the
parquet need to be aware of this difference.

**Pitfall 5: Duplicate fiscal years after PIT filter**
A company may file an amendment (10-K/A) for the same fiscal year, creating two rows in
SF1 with the same `calendardate` but different `datekeys`. Always deduplicate by taking
the most recent `datekey ≤ signal_date` per ticker × fiscal year before counting observations.


## §14. Final summary — what "done" looks like

**The deliverable is one file:** `data/out/gro_use4.parquet`

**Acceptance criteria:**

1. ✅ Schema matches §3 exactly (7 columns, types as specified)
2. ✅ All 6 required validation checks in §11 pass within tolerance
3. ✅ ESTU has ~2,500–3,500 names per date, stable over time
4. ✅ `|cap-weighted mean of gro_score| < 1e-6` for every ESTU date
5. ✅ `|EW std of gro_score − 1.0| < 0.02` for every ESTU date
6. ✅ Raw `gro` has median ≈ 0.05–0.30 and max |gro| < 10.0 post-clip
7. ✅ Coverage ≥ 4,000 stocks per date post-2005
8. ✅ Month-over-month rank stability ρ > 0.95
9. ✅ All ❓ NOT-IN-PDF decisions in §12 are documented in the code comments

**Known deviations from USE4 (must be documented in code):**
- EGRLF proxied by EGRO (Sharadar lacks analyst LTG forecasts) → effective weights 0.90·EGRO + 0.10·SGRO
- Stocks with negative mean EPS excluded from EGRO (null rather than sign-flipped)
- Minimum 3 annual observations accepted (USE4 targets 5)

**Once GRO is built and passes validation, the next steps are:**
- GRO has no known downstream consumers in the current USE4 build pipeline
- Remaining USE4 factors to build: Size, Liquidity, B/P (Value), Leverage, Dividend Yield
